# Model Experiments

1. Logistic Regression": LogisticRegression(),
2. Random Forest": RandomForestClassifier(),
3. XGBoost": XGBClassifier(),
4. CatBoost": CatBoostClassifier()

### Chainging the directory path

In [ ]:
import os

In [ ]:
%pwd

In [ ]:
os.chdir('../')
%pwd

### Loading the required library

In [ ]:
# pip install catboost

In [47]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


### Loading Train and valid data of X and y

In [48]:
X_train = np.load('/content/X_train.npy')
print(X_train.shape)
X_valid = np.load('/content/X_valid.npy')
print(X_valid.shape)


y_train = np.load('/content/y_train.npy')
print(y_train.shape)
y_valid = np.load('/content/y_valid.npy')
print(y_valid.shape)

(13526, 6)
(1500, 6)
(13526,)
(1500,)


### Initalizing Models

In [49]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
}

### Tunning Parameters for each models

In [50]:
#Hyper parameter tunning
params = {

    "Random Forest": {
        "n_estimators": [100, 200, 500],
        "max_depth": [None, 5, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"],
        #"class_weight": [None, "balanced"]
    },

    "Logistic Regression": {
        "C": [0.001, 0.01, 0.1, 1, 10, 100],
        "penalty": ["l1", "l2"],
        "solver": ["liblinear", "saga"],
        #"class_weight": [None, "balanced"]
    },

    "XGBoost": {
        "n_estimators": [100, 200, 500],
        "max_depth": [3, 5, 7, 10],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "gamma": [0, 0.1, 0.3],
        "min_child_weight": [1, 3, 5],
        #"scale_pos_weight": [1, 10, 20, 28]
    },

    "CatBoost": {
        "iterations": [100, 300, 500],
        "depth": [4, 6, 8, 10],
        "learning_rate": [0.01, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 5, 7],
        "border_count": [32, 64, 128]
    }
}

### Training the model

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import joblib

results = []
best_model = None
best_model_name = None
best_validation_f1 = -1

for model_name, model in models.items():

    print(f"\n{'='*50}")
    print(f"Tuning {model_name}")
    print(f"{'='*50}")

    # HYPER PARAMETER :- RandomizedSearchCV
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params[model_name],
        n_iter=5, #20
        cv=3, # 5
        scoring="f1",
        n_jobs=-1,
        verbose=1,
        random_state=42 #,
        #error_score="raise"
    )

    random_search.fit(X_train, y_train)

    search_results_df = pd.DataFrame(random_search.cv_results_)

    os.makedirs("artifacts/models/search_results", exist_ok=True)

    safe_model_name = model_name.lower().replace(" ", "_")

    search_results_df.to_csv(
        f"artifacts/models/search_results/{safe_model_name}_search_results.csv",
        index=False
    )

    current_best_model = random_search.best_estimator_

    y_pred = current_best_model.predict(X_valid)

    if hasattr(current_best_model, "predict_proba"):
        y_prob = current_best_model.predict_proba(X_valid)[:, 1]
        roc_auc = roc_auc_score(y_valid, y_prob)
    else:
        roc_auc = np.nan

    validation_f1 = f1_score(y_valid, y_pred)

    results.append({
        "Model": model_name,
        "Best CV F1": round(random_search.best_score_, 4),
        "Validation Accuracy": round(accuracy_score(y_valid, y_pred), 4),
        "Validation Precision": round(precision_score(y_valid, y_pred), 4),
        "Validation Recall": round(recall_score(y_valid, y_pred), 4),
        "Validation F1": round(validation_f1, 4),
        "Validation ROC-AUC": round(roc_auc, 4),
        "Best Params": random_search.best_params_
    })

    if validation_f1 > best_validation_f1:
        best_validation_f1 = validation_f1
        best_model = current_best_model
        best_model_name = model_name
        best_params = random_search.best_params_


Tuning Logistic Regression
Fitting 3 folds for each of 5 candidates, totalling 15 fits

Tuning Random Forest
Fitting 3 folds for each of 5 candidates, totalling 15 fits

Tuning XGBoost
Fitting 3 folds for each of 5 candidates, totalling 15 fits

Tuning CatBoost
Fitting 3 folds for each of 5 candidates, totalling 15 fits


### Model Comparison

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
      by="Validation F1",
      ascending=False
    ).reset_index(drop=True)

print("\nModel Comparison")
print(results_df)

print(f"\nBest Model: {best_model_name}")
print(f"Best Validation F1: {best_validation_f1:.4f}")

os.makedirs("artifacts/models", exist_ok=True)
joblib.dump(best_model, "artifacts/models/best_model.pkl")

print("\nBest model saved as best_model.pkl")


Model Comparison
                 Model  Best CV F1  Validation Accuracy  Validation Precision  \
0              XGBoost      0.9858                0.980                0.6780   
1             CatBoost      0.9858                0.976                0.6316   
2        Random Forest      0.9802                0.968                0.5224   
3  Logistic Regression      0.8251                0.842                0.1450   

   Validation Recall  Validation F1  Validation ROC-AUC  \
0             0.7843         0.7273              0.9814   
1             0.7059         0.6667              0.9757   
2             0.6863         0.5932              0.9789   
3             0.7451         0.2428              0.8777   

                                         Best Params  
0  {'subsample': 0.8, 'n_estimators': 500, 'min_c...  
1  {'learning_rate': 0.1, 'l2_leaf_reg': 5, 'iter...  
2  {'n_estimators': 100, 'min_samples_split': 5, ...  
3  {'solver': 'liblinear', 'penalty': 'l1', 'C': 10}  

Best

### Saving Model Report

In [ ]:
import yaml

In [ ]:
os.makedirs("artifacts/models", exist_ok=True)

with open("artifacts/models/model_report.yaml", "w") as file:
    yaml.dump(
    results_df.to_dict(orient="records"),
    file
)

### Saving the best Params

In [ ]:
os.makedirs("artifacts/models", exist_ok=True)

best_model_info = {
    "model_name": best_model_name,
    "validation_f1": float(best_validation_f1),
    "best_params": best_params
}

with open("artifacts/models/best_params.yaml", "w") as file:
    yaml.dump(
        best_model_info,
        file,
        default_flow_style=False
    )

### Confusion Matrix

In [53]:
from sklearn.metrics import confusion_matrix

best_y_pred = best_model.predict(X_valid)

cm = confusion_matrix(
    y_valid,
    best_y_pred
)
print(cm)

[[1430   19]
 [  11   40]]


### Classification Report

In [54]:
from sklearn.metrics import classification_report

print(classification_report(y_valid, best_y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1449
           1       0.68      0.78      0.73        51

    accuracy                           0.98      1500
   macro avg       0.84      0.89      0.86      1500
weighted avg       0.98      0.98      0.98      1500



### Loading the test data 

In [55]:
X_test = np.load('/content/X_test.npy')
print(X_test.shape)

(1500, 6)


### Making Prediction on test data

In [ ]:
best_model = joblib.load(
    "artifacts/models/best_model.pkl"
)
y_test_pred = best_model.predict(X_test)

In [57]:
y_test = np.load('/content/y_test.npy')
print(y_test.shape)

(1500,)


### Evaluating the best model

In [58]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# Class predictions
y_test_pred = best_model.predict(X_test)

# Probability predictions (needed for ROC-AUC)
y_test_prob = best_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
roc_auc = roc_auc_score(y_test, y_test_prob)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

Accuracy : 0.9740
Precision: 0.5811
Recall   : 0.8431
F1 Score : 0.6880
ROC-AUC  : 0.9798


Recall = 84.31%

Your model catches about 84 out of every 100 actual failures.

For predictive maintenance, this is usually the most important metric because missed failures are expensive.

Precision = 58.11%

When the model predicts a failure, it is correct about 58% of the time.

In other words, there are some false alarms, but not an unreasonable amount.

F1 = 0.688

This is a solid balance between precision and recall on an imbalanced classification problem.

ROC-AUC = 0.9798

This is excellent.

It means the model is very good at ranking failure cases above non-failure cases.

# The final XGBoost model achieved:


Accuracy: 97.40%

Precision: 58.11%

Recall: 84.31%

F1 Score: 68.80%

ROC-AUC: 97.98%

The model successfully identifies the majority of machine failure events while maintaining strong overall discriminative performance.

The model training phase is complete and successful. Multiple candidate models were evaluated using randomized hyperparameter search, with XGBoost selected as the final model based on validation F1 score. The chosen model demonstrates strong predictive performance on an imbalanced predictive maintenance dataset, achieving approximately 84% recall and 98% ROC-AUC on the held-out test set. The next logical step is model evaluation, experiment tracking, and deployment integration rather than further hyperparameter tuning.